# Supplementary evidence on the concept labels of Chanda et al. (2024)

We collect in this notebook the analyses behind our Matters Arising that did not fit in its 1,200 words. In each section, we state the question, compute the evidence, and close with a conclusion that we check with assertions.

We keep the scope of every conclusion narrow. We show how the annotation protocol, the union rule and the annotators shaped the concept labels. We do not claim that the concept labels fail to capture the appearance of the lesions.

The notebook reads the Figshare files from the data folder of this package, or from the path in the environment variable CHANDA_DATA, and unzips Datasets.zip when needed.

In [1]:
import os
import itertools
import zipfile
import numpy as np
import pandas as pd
from scipy import stats

ROOT = os.environ.get("CHANDA_DATA", os.path.join("..", "data"))
if not os.path.isdir(os.path.join(ROOT, "Datasets")):
    with zipfile.ZipFile(os.path.join(ROOT, "Datasets.zip")) as z:
        z.extractall(ROOT)

gt = pd.read_csv(os.path.join(ROOT, "metadata_gt.csv"))
ann = pd.read_csv(os.path.join(ROOT, "Datasets", "metadata", "metadata_ground_truth.csv"))
study = pd.read_csv(os.path.join(ROOT, "Datasets", "metadata", "metadata_study.csv"))

# Concept codes of the ontology, grouped by the pool of each class
M = ["TRBL", "ESA", "BDG", "GP", "PV", "PRL", "WLSA", "PLR", "PES", "PIF"]
N = ["OPC", "SPC", "MVP", "PRLC", "PLF", "PDES", "APC", "MS"]
POOL = M + N
ALL = POOL + ["None"]
NAME = {"TRBL": "Thick reticular or branched lines", "ESA": "Eccentric structureless area",
        "BDG": "Black dots or globules in periphery", "GP": "Grey patterns", "PV": "Polymorphous vessels",
        "PRL": "Pseudopods or radial lines (partial)", "WLSA": "White lines or white structureless area",
        "PLR": "Parallel lines on ridges", "PES": "Pigmentation beyond scar", "PIF": "Pigmentation invades follicles",
        "OPC": "Only one pattern and one color", "SPC": "Symmetrical combination", "MVP": "Monomorphic vascular pattern",
        "PRLC": "Pseudopods or radial lines (entire)", "PLF": "Parallel lines in furrows",
        "PDES": "Pigmentation confined to scar", "APC": "Asymmetric combination", "MS": "Melanoma simulator",
        "None": "None option"}
POOL_OF = {**{f: "M" for f in M}, **{f: "N" for f in N}, "None": "-"}
Y = gt["benign_malignant"].to_numpy()
n, n1 = len(gt), int(Y.sum())
n0 = n - n1


def kappa(x, y):
    # Cohen's kappa for two binary raters
    x, y = np.asarray(x, float), np.asarray(y, float)
    po = (x == y).mean()
    pe = x.mean() * y.mean() + (1 - x.mean()) * (1 - y.mean())
    return np.nan if pe == 1 else (po - pe) / (1 - pe)


def entropy(p):
    # Shannon entropy in nats of a discrete distribution
    p = np.array([q for q in p if q > 0])
    return float(-(p * np.log(p)).sum())


print(n, n1, n0)

3611 1113 2498


## 1. Annotation coverage

Chanda et al. state that at least two dermatologists annotated every unique lesion. We count the annotators behind each image and each lesion.

In [2]:
per_img = ann.groupby("image_id").annotator.nunique()
per_les = ann.groupby("lesion_id").annotator.nunique()
single_les = per_les[per_les == 1].index
n_single_img = gt.lesion_id.isin(single_les).sum()
print("records per annotator")
print(ann.annotator.value_counts().sort_index().to_dict())
print("annotators per image", per_img.value_counts().sort_index().to_dict())
print("lesions with a single annotator", len(single_les), "covering", n_single_img, "images")
assert (per_img == 1).sum() == 995 and len(single_les) == 19 and n_single_img == 30

records per annotator
{1: 3611, 2: 438, 3: 215, 4: 163, 5: 196, 6: 222, 7: 210, 8: 222, 9: 199, 10: 217, 11: 186, 12: 199, 13: 200, 14: 220}
annotators per image {1: 995, 2: 2345, 3: 271}
lesions with a single annotator 19 covering 30 images


**Conclusion.** Annotator 1 annotated all 3,611 images, and annotators 2 to 14 annotated between 163 and 438 images each. We find 995 images (27.6%) with the concept labels of annotator 1 alone. We also find 19 lesions (30 images) with a single annotator, which contradicts the statement that at least two dermatologists annotated every unique lesion.

## 2. Prevalence and class exclusivity

We compute the prevalence of positive concept labels for each concept and for the None option within each class after the union rule. A pool concept is class-exclusive when it never appears in the other class.

In [3]:
prev = pd.DataFrame({
    "concept": [NAME[f] for f in ALL],
    "pool": [POOL_OF[f] for f in ALL],
    "melanoma %": [round(100 * gt.loc[Y == 1, f].mean(), 1) for f in ALL],
    "nevus %": [round(100 * gt.loc[Y == 0, f].mean(), 1) for f in ALL]}, index=ALL)
exclusive = [(gt.loc[Y == 0, f].sum() == 0) if f in M else (gt.loc[Y == 1, f].sum() == 0) for f in POOL]
assert all(exclusive)
assert gt.loc[Y == 1, "None"].sum() > 0 and gt.loc[Y == 0, "None"].sum() > 0
prev

,concept,pool,melanoma %,nevus %
TRBL,Thick reticular or branched lines,M,90.5,0.0
ESA,Eccentric structureless area,M,53.9,0.0
BDG,Black dots or globules in periphery,M,81.9,0.0
GP,Grey patterns,M,94.1,0.0
PV,Polymorphous vessels,M,47.9,0.0
PRL,Pseudopods or radial lines (partial),M,25.9,0.0
WLSA,White lines or white structureless area,M,50.6,0.0
PLR,Parallel lines on ridges,M,0.9,0.0
PES,Pigmentation beyond scar,M,1.1,0.0
PIF,Pigmentation invades follicles,M,6.6,0.0


**Conclusion.** Each of the 18 pool concepts appears only in its own class, whereas the None option appears in both classes. Some merged prevalences are very high, such as grey patterns in 94.1% of melanomas and melanoma simulator in 97.8% of nevi.

## 3. Pairwise association between concepts

We compute the phi coefficient for all 171 pairs among the 18 concepts and the None option and count the pairs that never co-occur. Two concepts from different pools can never co-occur, because annotators never combined the pools.

In [4]:
rows = []
for a, b in itertools.combinations(ALL, 2):
    x, y = gt[a].to_numpy(), gt[b].to_numpy()
    rows.append(dict(A=a, B=b, pools=POOL_OF[a] + POOL_OF[b], n11=int(((x == 1) & (y == 1)).sum()), phi=np.corrcoef(x, y)[0, 1]))
pairs = pd.DataFrame(rows)
zero = pairs[pairs.n11 == 0]
kind = zero.pools.map(lambda s: "across pools" if set(s) == {"M", "N"} else ("within melanoma pool" if s == "MM" else ("within nevus pool" if s == "NN" else "with None")))
print("pairs", len(pairs), "zero co-occurrence", len(zero))
print(kind.value_counts().to_dict())
assert len(zero) == 89 and (kind == "across pools").sum() == 80
top = pairs.reindex(pairs.phi.abs().sort_values(ascending=False).index).head(20)
assert (top.loc[top.phi < 0, "pools"].map(set) == {"M", "N"}).all()
top.assign(phi=top.phi.round(3))

pairs 171 zero co-occurrence 89
{'across pools': 80, 'within melanoma pool': 5, 'within nevus pool': 3, 'with None': 1}


,A,B,pools,n11,phi
64,GP,MS,MN,0,-0.924
16,TRBL,MS,MN,0,-0.899
2,TRBL,GP,MM,951,0.897
35,BDG,GP,MM,869,0.849
1,TRBL,BDG,MM,848,0.844
49,BDG,MS,MN,0,-0.840
168,APC,MS,NN,2042,0.770
63,GP,APC,MN,0,-0.740
15,TRBL,APC,MN,0,-0.720
48,BDG,APC,MN,0,-0.673


**Conclusion.** We find 89 of the 171 pairs with zero co-occurrence. Of these, 80 pairs join a melanoma concept with a nevus concept and vanish by construction of the pools. The remaining nine pairs involve rare concepts, namely five melanoma pairs, three nevus pairs and pigmentation confined to scar with the None option. Every negative value among the twenty largest |phi| values comes from a pair across pools. For this reason, we attribute these values to the protocol rather than to any incompatibility in appearance.

## 4. Redundancy within the melanoma pool

A large positive phi between two melanoma concepts could signal that one concept duplicates the other. We cross-tabulate four strongly associated pairs within melanoma images only.

In [5]:
mel = gt[Y == 1]
rows = []
for a, b in [("TRBL", "ESA"), ("TRBL", "WLSA"), ("GP", "WLSA"), ("GP", "PV")]:
    x, y = mel[a], mel[b]
    rows.append(dict(A=NAME[a], B=NAME[b], both=100 * (x & y).mean(), only_A=100 * (x & (1 - y)).mean(),
                     only_B=100 * ((1 - x) & y).mean(), neither=100 * ((1 - x) & (1 - y)).mean()))
cond = pd.DataFrame(rows).round(1)
assert (cond.only_A > 20).all()
cond

,A,B,both,only_A,only_B,neither
0,Thick reticular or branched lines,Eccentric structureless area,47.9,42.6,6.0,3.5
1,Thick reticular or branched lines,White lines or white structureless area,43.9,46.5,6.6,2.9
2,Grey patterns,White lines or white structureless area,48.2,45.9,2.4,3.5
3,Grey patterns,Polymorphous vessels,46.7,47.3,1.2,4.8


**Conclusion.** For all four pairs, a large share of melanoma images carries the first concept without the second. Hence, we attribute most of the dataset-level phi of these pairs to shared class membership, and we find no sign that one concept of a pair duplicates the other.

## 5. Association between each concept and the diagnosis

For two binary variables, the point-biserial correlation equals phi and the Pearson statistic without continuity correction equals n times phi squared. We apply a Bonferroni correction over the 18 concepts and the None option.

In [6]:
rows = []
for f in ALL:
    x = gt[f].to_numpy()
    phi = np.corrcoef(x, Y)[0, 1]
    chi2, p, _, _ = stats.chi2_contingency(pd.crosstab(x, Y).values, correction=False)
    rows.append(dict(concept=NAME[f], phi=phi, chi2=chi2, n_phi2=n * phi ** 2, p=p))
assoc = pd.DataFrame(rows, index=ALL)
assoc = assoc.reindex(assoc.phi.abs().sort_values(ascending=False).index)
assert np.allclose(assoc.chi2, assoc.n_phi2)
alpha = 0.05 / len(ALL)
not_sig = assoc.index[assoc.p > alpha].tolist()
print("Bonferroni threshold", round(alpha, 4), "not significant", not_sig)
assert not_sig == ["PDES"]
assoc.style.format({"phi": "{:.3f}", "chi2": "{:.1f}", "n_phi2": "{:.1f}", "p": "{:.2e}"})

Bonferroni threshold 0.0026 not significant ['PDES']


,concept,phi,chi2,n_phi2,p
MS,Melanoma simulator,-0.965,3360.9,3360.9,0.00e+00
GP,Grey patterns,0.957,3309.4,3309.4,0.00e+00
TRBL,Thick reticular or branched lines,0.932,3134.1,3134.1,0.00e+00
BDG,Black dots or globules in periphery,0.871,2738.5,2738.5,0.00e+00
APC,Asymmetric combination,-0.773,2156.3,2156.3,0.00e+00
ESA,Eccentric structureless area,0.669,1615.0,1615.0,0.00e+00
WLSA,White lines or white structureless area,0.644,1497.0,1497.0,0.00e+00
PV,Polymorphous vessels,0.623,1403.4,1403.4,3.82e-307
PRL,Pseudopods or radial lines (partial),0.441,702.4,702.4,8.97e-155
SPC,Symmetrical combination,-0.355,455.7,455.7,4.18e-101


**Conclusion.** Every concept except pigmentation confined to scar passes the Bonferroni threshold, and so does the None option, whereas that concept reaches p = 0.035. The pools tie each concept to one class, so these associations cannot show by themselves that the concepts separate the classes by appearance.

## 6. Entropy and mutual information

We compute the entropy of the concept labels of each concept overall and within each class, in nats, and the mutual information with the diagnosis, in bits. A concept whose concept labels have zero entropy within a class carries no image-specific information in that class.

In [7]:
rows = []
for f in ALL:
    x = gt[f].to_numpy()
    h = entropy([x.mean(), 1 - x.mean()])
    hm = entropy([x[Y == 1].mean(), 1 - x[Y == 1].mean()])
    hn = entropy([x[Y == 0].mean(), 1 - x[Y == 0].mean()])
    rows.append(dict(concept=NAME[f], pool=POOL_OF[f], H=h, H_mel=hm, H_nev=hn, I_bits=(h - (n1 / n) * hm - (n0 / n) * hn) / np.log(2)))
info = pd.DataFrame(rows, index=ALL)
assert (info.loc[M, "H_nev"] == 0).all() and (info.loc[N, "H_mel"] == 0).all()
info.round(3)

,concept,pool,H,H_mel,H_nev,I_bits
TRBL,Thick reticular or branched lines,M,0.592,0.314,-0.000,0.714
ESA,Eccentric structureless area,M,0.450,0.690,-0.000,0.342
BDG,Black dots or globules in periphery,M,0.565,0.472,-0.000,0.605
GP,Grey patterns,M,0.602,0.225,-0.000,0.769
PV,Polymorphous vessels,M,0.419,0.692,-0.000,0.296
PRL,Pseudopods or radial lines (partial),M,0.278,0.572,-0.000,0.147
WLSA,White lines or white structureless area,M,0.433,0.693,-0.000,0.316
PLR,Parallel lines on ridges,M,0.019,0.051,-0.000,0.005
PES,Pigmentation beyond scar,M,0.022,0.060,-0.000,0.006
PIF,Pigmentation invades follicles,M,0.100,0.244,-0.000,0.035


**Conclusion.** Every melanoma concept has zero entropy within nevi, and every nevus concept has zero entropy within melanomas. As a result, part of the mutual information between each pool concept and the diagnosis comes from the protocol itself. The pattern-complexity concepts differ from the melanoma concepts in meaning rather than in these statistics. Asymmetry and color variation can occur in melanomas, yet annotators never selected these concepts for a melanoma.

## 7. Visual characteristics split across the two pools

Four characteristics appear with one level in the melanoma pool and the other level in the nevus pool. We report the prevalence of each level within each class.

In [8]:
split = [("PRL", "PRLC"), ("PLR", "PLF"), ("PES", "PDES"), ("PV", "MVP")]
rows = []
for m, v in split:
    rows.append(dict(melanoma_level=NAME[m], nevus_level=NAME[v],
                     mel_level_in_mel=round(100 * gt.loc[Y == 1, m].mean(), 1), mel_level_in_nev=round(100 * gt.loc[Y == 0, m].mean(), 1),
                     nev_level_in_mel=round(100 * gt.loc[Y == 1, v].mean(), 1), nev_level_in_nev=round(100 * gt.loc[Y == 0, v].mean(), 1)))
lev = pd.DataFrame(rows)
assert (lev.mel_level_in_nev == 0).all() and (lev.nev_level_in_mel == 0).all()
lev

,melanoma_level,nevus_level,mel_level_in_mel,mel_level_in_nev,nev_level_in_mel,nev_level_in_nev
0,Pseudopods or radial lines (partial),Pseudopods or radial lines (entire),25.9,0.0,0.0,10.3
1,Parallel lines on ridges,Parallel lines in furrows,0.9,0.0,0.0,2.6
2,Pigmentation beyond scar,Pigmentation confined to scar,1.1,0.0,0.0,0.4
3,Polymorphous vessels,Monomorphic vascular pattern,47.9,0.0,0.0,3.8


**Conclusion.** For each of the four characteristics, annotators recorded only the level that agrees with the known diagnosis. A concept model trained on these concept labels never observes either level within the other class, so each level also marks the class.

## 8. Agreement between annotator 1 and the other annotators

We pair each record of annotators 2 to 14 with the record of annotator 1 for the same image. For every concept and the None option, we report kappa on all pairs, kappa and observed agreement within the class of the pool, positive agreement, and the prevalence for both rater groups.

In [9]:
a1 = ann[ann.annotator == 1].set_index("image_id")[ALL]
pr = ann[ann.annotator != 1].join(a1, on="image_id", rsuffix="_a1")
rows = []
for f in ALL:
    cls = 1 if f in M else (0 if f in N else None)
    sub = pr if cls is None else pr[pr.benign_malignant == cls]
    x, y = sub[f].to_numpy(), sub[f + "_a1"].to_numpy()
    both = ((x == 1) & (y == 1)).sum()
    den = 2 * both + ((x == 1) & (y == 0)).sum() + ((x == 0) & (y == 1)).sum()
    rows.append(dict(concept=NAME[f], kappa_all=kappa(pr[f], pr[f + "_a1"]), kappa_within=kappa(x, y),
                     observed_within=(x == y).mean(), positive_agreement=2 * both / den if den else np.nan,
                     prev_a1=100 * y.mean(), prev_others=100 * x.mean()))
agree = pd.DataFrame(rows, index=ALL)
assert len(pr) == 2887
assert agree.loc[["MS", "GP", "TRBL", "APC", "BDG"], "observed_within"].between(0.25, 0.46).all()
agree.round(3)

,concept,kappa_all,kappa_within,observed_within,positive_agreement,prev_a1,prev_others
TRBL,Thick reticular or branched lines,0.399,0.080,0.408,0.519,90.516,32.363
ESA,Eccentric structureless area,0.333,0.165,0.605,0.417,22.712,45.092
BDG,Black dots or globules in periphery,0.390,0.127,0.458,0.498,78.785,29.118
GP,Grey patterns,0.352,0.029,0.345,0.476,94.592,30.449
PV,Polymorphous vessels,0.342,0.226,0.637,0.406,45.424,15.641
PRL,Pseudopods or radial lines (partial),0.308,0.242,0.808,0.349,11.897,17.637
WLSA,White lines or white structureless area,0.501,0.337,0.689,0.576,31.697,41.681
PLR,Parallel lines on ridges,0.879,0.879,0.998,0.880,0.998,1.082
PES,Pigmentation beyond scar,0.399,0.397,0.993,0.400,0.333,0.915
PIF,Pigmentation invades follicles,0.303,0.293,0.944,0.309,1.414,6.656


**Conclusion.** For the five most frequent concepts, kappa within the class ranges from 0.01 to 0.13, and observed agreement within the class ranges from 26% to 46%. Hence, we can rule out the kappa paradox, which concerns high observed agreement with low kappa. Agreement is good only for the rare concept parallel lines on ridges (kappa 0.879).

## 9. Prevalence by annotator

We compute the share of images for which each annotator selected melanoma simulator among nevi and grey patterns among melanomas.

In [10]:
nv = ann[ann.benign_malignant == 0]
mv = ann[ann.benign_malignant == 1]
by_annot = pd.DataFrame({
    "nevus records": nv.groupby("annotator").size(),
    "melanoma simulator % of nevi": (100 * nv.groupby("annotator").MS.mean()).round(1),
    "melanoma records": mv.groupby("annotator").size(),
    "grey patterns % of melanomas": (100 * mv.groupby("annotator").GP.mean()).round(1)})
others = by_annot.drop(index=1)
print("annotator 1", by_annot.loc[1].to_dict())
print("others, melanoma simulator range", others["melanoma simulator % of nevi"].min(), "to", others["melanoma simulator % of nevi"].max())
print("others, grey patterns range", others["grey patterns % of melanomas"].min(), "to", others["grey patterns % of melanomas"].max())
assert by_annot.loc[1, "melanoma simulator % of nevi"] == 97.7 and others["melanoma simulator % of nevi"].max() < 60
by_annot

annotator 1 {'nevus records': 2498.0, 'melanoma simulator % of nevi': 97.7, 'melanoma records': 1113.0, 'grey patterns % of melanomas': 93.5}
others, melanoma simulator range 2.1 to 58.2
others, grey patterns range 1.8 to 78.1


,nevus records,melanoma simulator % of nevi,melanoma records,grey patterns % of melanomas
annotator,,,,
1,2498,97.7,1113,93.5
2,320,26.6,118,44.9
3,97,2.1,118,24.6
4,69,20.3,94,22.3
5,105,25.7,91,26.4
6,114,29.8,108,58.3
7,117,11.1,93,21.5
8,158,46.8,64,78.1
9,135,10.4,64,7.8


**Conclusion.** Annotator 1 selected melanoma simulator for 97.7% of nevi, whereas the other annotators selected it for between 2.1% and 58.2% of their nevi. Annotator 1 also selected grey patterns for 93.5% of melanomas, against 1.8% to 78.1% for the other annotators. Chanda et al. acknowledge the first pattern in their Discussion but do not report the second.

## 10. Share of merged positive concept labels that rest on annotator 1 alone

Under the union rule, a merged positive concept label rests on annotator 1 alone when no other annotator of that image selected the concept. We compute this share for the frequent concepts.

In [11]:
others_pos = ann[ann.annotator != 1].groupby("image_id")[ALL].max().reindex(a1.index).fillna(0)
rows = []
for f in ["MS", "APC", "GP", "BDG", "TRBL"]:
    only = int(((a1[f] == 1) & (others_pos[f] == 0)).sum())
    rows.append(dict(concept=NAME[f], merged_positives=int(gt[f].sum()), annotator1_alone=only, share=round(100 * only / gt[f].sum(), 1)))
dom = pd.DataFrame(rows)
assert dom.share.min() > 60
dom

,concept,merged_positives,annotator1_alone,share
0,Melanoma simulator,2442,2030,83.1
1,Asymmetric combination,2068,1638,79.2
2,Grey patterns,1047,709,67.7
3,Black dots or globules in periphery,912,599,65.7
4,Thick reticular or branched lines,1007,646,64.2


**Conclusion.** For each of the five most frequent concepts, annotator 1 alone supplies between 64.2% and 83.1% of the merged positive concept labels. We conclude that the merged prevalence of these concepts follows the habits of annotator 1.

## 11. Contradictory pattern-complexity states

The three pattern-complexity concepts describe exclusive states of one characteristic. We count contradictory pairs within single records and after the union rule.

In [12]:
rows = []
for a, b in [("OPC", "APC"), ("SPC", "APC"), ("OPC", "SPC")]:
    within = int(((ann[a] == 1) & (ann[b] == 1)).sum())
    after = int(((gt[a] == 1) & (gt[b] == 1)).sum())
    rows.append(dict(A=NAME[a], B=NAME[b], within_records=within, merged_images=after, union_share_upper_bound=round(100 * (after - within) / after, 1)))
pcx = pd.DataFrame(rows)
assert pcx.within_records.tolist() == [6, 22, 26] and pcx.merged_images.tolist() == [297, 446, 129]
pcx

,A,B,within_records,merged_images,union_share_upper_bound
0,Only one pattern and one color,Asymmetric combination,6,297,98.0
1,Symmetrical combination,Asymmetric combination,22,446,95.1
2,Only one pattern and one color,Symmetrical combination,26,129,79.8


**Conclusion.** Contradictory states are rare within single records but common after the union rule. Chanda et al. thus turned disagreement between annotators into contradictory states of one image when they merged the records. We report the union share as an upper bound, because a contradiction inside one record also survives the union.

## 12. The None option

Chanda et al. do not define the None option in their documentation. We examine how annotators used it and what the union rule did with it.

In [13]:
nr = ann[ann["None"] == 1]
k = nr[POOL].sum(axis=1)
none_only = nr[k == 0]
print("None records", len(nr), "with a pool concept in the same record", int((k > 0).sum()), "None alone", len(none_only))
print("None-only records by annotator", none_only.annotator.value_counts().to_dict())
print("None-only melanoma records", int((none_only.benign_malignant == 1).sum()))
ni = gt[gt["None"] == 1]
kk = ni[POOL].sum(axis=1)
dist = pd.crosstab(kk.rename("other concepts"), ni.benign_malignant.map({1: "melanoma", 0: "nevus"}), margins=True, margins_name="total")
assert len(nr) == 373 and (k > 0).sum() == 303 and len(none_only) == 70
assert none_only.annotator.isin([3, 4, 5]).sum() == 60 and (kk == 0).sum() == 0
print("mean number of other concepts", round(kk.mean(), 2), round(kk[ni.benign_malignant == 1].mean(), 2), round(kk[ni.benign_malignant == 0].mean(), 2))
dist

None records 373 with a pool concept in the same record 303 None alone 70
None-only records by annotator {3: 30, 4: 17, 5: 13, 6: 2, 9: 2, 11: 2, 12: 2, 13: 2}
None-only melanoma records 46
mean number of other concepts 4.0 4.65 2.15


benign_malignant,melanoma,nevus,total
other concepts,,,
1,1,17,18
2,10,50,60
3,37,19,56
4,59,6,65
5,94,0,94
6,47,0,47
7,14,0,14
total,262,92,354


**Conclusion.** Annotators selected None in 373 records, and 303 of these records also contain a pool concept from the same annotator. The remaining 70 records contain None alone, and 60 of them come from annotators 3, 4 and 5. In 46 of these records, an annotator who knew the lesion was a melanoma selected no melanoma concept. After the union rule, every one of the 354 images with None also carries a pool concept, so the merged concept labels no longer show these judgments.

## 13. Clinicians without knowledge of the histopathological diagnosis

In phase 1 of the reader study, clinicians selected concepts before they saw any AI output. We compare their use of melanoma simulator with the merged concept labels.

In [14]:
S = (study[ALL] == 1).astype(int)
dx_mel = (study.prediction == 1).to_numpy()
ms_rate = round(100 * S.loc[~dx_mel, "MS"].mean(), 1)
print("melanoma simulator among nevus diagnoses of clinicians (%)", ms_rate, "against", round(100 * gt.loc[Y == 0, "MS"].mean(), 1), "in the merged labels")
assert ms_rate == 18.3

melanoma simulator among nevus diagnoses of clinicians (%) 18.3 against 97.8 in the merged labels


**Conclusion.** Clinicians who diagnosed a nevus without knowing the histopathological diagnosis selected melanoma simulator in 18.3% of their records, against 97.8% of nevi in the merged concept labels. Hence, the use of this concept depends strongly on the person who selects it.

## 14. Overlap between the released training split and the test lesions

We check whether any training image in the released split belongs to a test lesion. The manuscript does not report this overlap, so we document it in this section for future users of the dataset.

In [15]:
test_les = gt.loc[gt.split == "test", "lesion_id"]
leak = gt[(gt.split != "test") & gt.lesion_id.isin(test_les)]
clean = gt[(gt.split == "train") & ~gt.lesion_id.isin(test_les)]
print("leaked images", len(leak), leak.benign_malignant.value_counts().to_dict(), "lesions", leak.lesion_id.nunique())
print("leakage-free training pool", len(clean), clean.benign_malignant.value_counts().to_dict())
assert len(leak) == 166 and leak.lesion_id.nunique() == 127 and len(clean) == 2646 + 599

leaked images 166 {1: 89, 0: 77} lesions 127
leakage-free training pool 3245 {0: 2321, 1: 924}


**Conclusion.** The released split column assigns 166 images (89 melanoma and 77 nevus images) from 127 test lesions to training. Removing them leaves 3,245 images, which equals the 2,646 training and 599 validation images that Chanda et al. report. We infer that the authors excluded these images, and we recommend that other users exclude them too.

## 15. Diagnostic accuracy of each concept label alone

Table 1 of the manuscript reports five diagnostic rules. We extend it to every pool concept, where a melanoma concept predicts melanoma when present and a nevus concept predicts melanoma when absent.

In [16]:
def bacc(pred, y):
    # Balanced accuracy of a binary melanoma prediction
    pred, y = np.asarray(pred), np.asarray(y)
    return 0.5 * ((pred[y == 1] == 1).mean() + (pred[y == 0] == 0).mean())


test = (gt.split == "test").to_numpy()
rows = []
for f in POOL:
    pred = gt[f].to_numpy() if f in M else 1 - gt[f].to_numpy()
    rows.append(dict(concept=NAME[f], pool=POOL_OF[f], all_images=round(bacc(pred, Y), 3), test_images=round(bacc(pred[test], Y[test]), 3)))
single = pd.DataFrame(rows).sort_values("all_images", ascending=False).reset_index(drop=True)
top5 = single.head(5)
assert (single.all_images.head(5) > 0.9).all() and top5.test_images.between(0.875, 0.98).all()
single

,concept,pool,all_images,test_images
0,Melanoma simulator,N,0.989,0.980
1,Grey patterns,M,0.970,0.980
2,Thick reticular or branched lines,M,0.952,0.955
3,Asymmetric combination,N,0.914,0.875
4,Black dots or globules in periphery,M,0.910,0.935
5,Eccentric structureless area,M,0.770,0.790
6,White lines or white structureless area,M,0.753,0.770
7,Polymorphous vessels,M,0.739,0.750
8,Symmetrical combination,N,0.660,0.755
9,Pseudopods or radial lines (partial),M,0.629,0.650


**Conclusion.** Rules based on a single concept reach a balanced accuracy above 0.90 on all images for five concepts, namely melanoma simulator, grey patterns, thick reticular lines, asymmetric combination and peripheral dots. On the 200 test images, rules based on these five concepts reach between 0.875 and 0.980, above the 0.81 that the XAI of Chanda et al. reached from the images. This comparison does not rank the methods, because annotators selected these concepts with knowledge of the diagnosis. It shows instead how much of the diagnosis the concept labels of each concept carry.

## 16. Overall conclusions

Chanda et al. asked dermatologists to annotate images from HAM10000 and gave each annotator the diagnosis together with a separate pool of concepts for each class. As a result, the merged concept labels determine the diagnosis exactly, alongside whatever they capture about the appearance of each lesion. Annotator 1 supplies most positive concept labels of the frequent concepts, and agreement with the other annotators within each class stays low.

The ontology also contains concepts that presuppose a benign diagnosis and concepts that describe mutually exclusive states, and the union rule turned disagreement between annotators into contradictory states. Clinicians without knowledge of the histopathological diagnosis also selected concepts only from the pool of their own diagnosis. For these reasons, we consider the dataset unsuitable for building or evaluating XAI methods that explain diagnoses through these concepts.

We recommend two revisions. First, the concept ontology needs concepts that are as independent of each other as possible, make no assumption about the diagnosis, and never contradict each other. Second, dermatologists need to reannotate the dataset with the whole set of concepts, so that every concept receives a concept label for every nevus and melanoma image. We also recommend that users exclude the 166 training images that belong to test lesions.